# QFIS — QLoRA Fine-tuning on Google Colab
**Model:** microsoft/phi-2 (2.7B)  
**Method:** QLoRA (4-bit NF4 quantization + LoRA)  
**Dataset:** FinQA (dreamerdeo/finqa)  
**Output:** Saved adapter weights → download → put in `models/v1/`

---
### Instructions
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run each cell **one by one** in order
3. After the last cell, **download `qfis_model_v1.zip`** from the Files panel
4. Extract it and place contents into your local `models/v1/` folder

## Step 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2 — Install Dependencies

In [ ]:
!pip install -q transformers==4.41.2 peft==0.11.1 bitsandbytes==0.43.1 \
    accelerate==0.30.1 datasets==2.19.1 trl einops loguru
print('Done installing')

## Step 3 — Load and Preprocess FinQA Dataset

In [ ]:
import json, re, hashlib
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def normalize_answer(text):
    text = str(text).lower().strip()
    text = re.sub(r'(\d),(\d)', r'\1\2', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def build_context(r):
    parts = []
    for field in ['pre_text', 'post_text']:
        val = r.get(field, [])
        if isinstance(val, list):
            parts.extend([p.strip() for p in val if isinstance(p, str) and p.strip()])
        elif isinstance(val, str) and val.strip():
            parts.append(val.strip())
    table = r.get('table', [])
    if isinstance(table, list):
        for row in table:
            if isinstance(row, list):
                parts.append(' | '.join(str(c) for c in row))
    return ' '.join(parts)[:1000]

def build_prompt(question, context):
    return f'Context: {context}\n\nQuestion: {question}\n\nAnswer:'

print('Loading FinQA...')
ds = load_dataset('dreamerdeo/finqa', trust_remote_code=True)

records = []
for split in ['train', 'validation', 'test']:
    if split in ds:
        for item in ds[split]:
            records.append(dict(item))
print(f'Raw records: {len(records)}')

# Clean
cleaned = []
for r in records:
    q = str(r.get('question', '')).strip()
    a = normalize_answer(r.get('answer', ''))
    ctx = build_context(r)
    if len(q) < 5 or not a or a in ('none','n/a','','nan','null') or len(ctx) < 10:
        continue
    cleaned.append({'question': q, 'answer': a, 'context': ctx,
                    'prompt': build_prompt(q, ctx), 'completion': a})

# Deduplicate
seen, unique = set(), []
for r in cleaned:
    key = hashlib.md5(r['question'].encode()).hexdigest()
    if key not in seen:
        seen.add(key)
        unique.append(r)

# Split 70/15/15
train_data, temp = train_test_split(unique, test_size=0.30, random_state=42)
val_data, test_data = train_test_split(temp, test_size=0.50, random_state=42)

# Cap for Colab T4 (15GB VRAM) — can use more than local
train_data = train_data[:2000]
val_data   = val_data[:400]

print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
print('Sample prompt:', train_data[0]['prompt'][:200])

## Step 4 — Load Phi-2 with 4-bit Quantization

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = 'microsoft/phi-2'
MAX_SEQ_LEN = 384

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading Phi-2 with 4-bit NF4 quantization...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Base model loaded')

## Step 5 — Apply LoRA Adapter

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'dense'],
    bias='none',
    inference_mode=False,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~10M trainable / 2.7B total (~0.37%)

## Step 6 — Tokenize Dataset

In [ ]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling

def build_hf_dataset(records):
    texts = [r['prompt'] + ' ' + r['completion'] + tokenizer.eos_token for r in records]
    def tokenize(batch):
        enc = tokenizer(batch['text'], truncation=True,
                        max_length=MAX_SEQ_LEN, padding='max_length')
        enc['labels'] = enc['input_ids'].copy()
        return enc
    ds = Dataset.from_dict({'text': texts})
    return ds.map(tokenize, batched=True, remove_columns=['text'])

print('Tokenizing...')
train_ds = build_hf_dataset(train_data)
val_ds   = build_hf_dataset(val_data)
print(f'Train tokens: {len(train_ds)} | Val tokens: {len(val_ds)}')

## Step 7 — Train

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    dataloader_num_workers=2,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Starting training... (T4: ~25-35 min)')
trainer.train()
print('Training complete!')

## Step 8 — Save Model Weights

In [ ]:
import json, os

SAVE_DIR = '/content/qfis_model_v1'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

meta = {
    'base_model': BASE_MODEL,
    'lora_r': 16,
    'lora_alpha': 32,
    'epochs': 3,
    'max_seq_len': MAX_SEQ_LEN,
    'quantization': '4-bit NF4',
    'train_samples': len(train_data),
    'trained_on': 'Google Colab T4',
}
with open(f'{SAVE_DIR}/training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Model saved to {SAVE_DIR}')
print('Files:', os.listdir(SAVE_DIR))

## Step 9 — Quick Inference Test (verify model works)

In [ ]:
from peft import PeftModel

model.eval()
test_prompt = 'Context: The company reported total revenue of $5.2 billion in 2022.\n\nQuestion: What was the total revenue in 2022?\n\nAnswer:'
inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=50, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)

new_tokens = out[0][inputs['input_ids'].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(f'Test answer: {answer}')
print('Model is working correctly!')

## Step 10 — Zip and Download

In [ ]:
import shutil
from google.colab import files

print('Zipping model...')
shutil.make_archive('/content/qfis_model_v1', 'zip', '/content/qfis_model_v1')
print('Downloading zip...')
files.download('/content/qfis_model_v1.zip')
print('Done! Extract and place contents into models/v1/ in your project.')

---
## After Download

1. Extract `qfis_model_v1.zip`
2. Copy all files into your local `models/v1/` folder
3. Restart the backend: `uvicorn app.main:app --host 0.0.0.0 --port 8000 --reload`
4. The API will auto-detect and load the fine-tuned weights
5. Run evaluation: `python training/evaluate.py`

**Expected files in models/v1/:**
```
adapter_config.json
adapter_model.safetensors
tokenizer.json
tokenizer_config.json
special_tokens_map.json
training_meta.json
```